# Import Libraries

In [1]:
import numpy as np
import pandas as pd
import pyhmmer

# Parse the HMM file using pyhmmer

In [2]:
"""
Load a HMMER3 .hmm file with pyhmmer and put the match emissions, insert
emissions, and transition probabilities into pandas DataFrames.

Row 0 = the model's BEGIN/insert-0 state; rows 1..M = the match nodes.
Columns for the emission tables are named after the model's canonical
alphabet symbols (e.g. A,C,D,E,... for amino acids). Values are already
plain probabilities (0-1), not log-odds scores.
"""

def hmm_to_dataframes(path: str):
    TRANS_COLS = ["MM", "MI", "MD", "IM", "II", "DM", "DD"]
    
    with pyhmmer.plan7.HMMFile(path) as f:
        hmm = f.read()

    # canonical residues are the first `K` symbols of the alphabet
    # (K = 20 for amino acids, 4 for DNA/RNA); the rest are gap/ambiguity codes
    k = hmm.alphabet.K
    residues = list(hmm.alphabet.symbols[:k])

    match_df = pd.DataFrame(
        np.asarray(hmm.match_emissions), columns=residues
    )
    match_df.index.name = "node"

    insert_df = pd.DataFrame(
        np.asarray(hmm.insert_emissions), columns=residues
    )
    insert_df.index.name = "node"

    trans_df = pd.DataFrame(
        np.asarray(hmm.transition_probabilities), columns=TRANS_COLS
    )
    trans_df.index.name = "node"

    return hmm, match_df, insert_df, trans_df   

# Create the Transition Matrix

In [3]:
def create_transition_matrix(transition_df):
    length = len(transition_df) - 1
    order = ['M', 'D', 'I']
    COLS = [order[j] + f'{i+1}' for i in range(length) for j in range (3)]
    COLS.insert(0, 'M0')
    COLS.insert(1, 'I0')
    COLS.append(f'M{length+1}')
    
    trans_matdf = pd.DataFrame(0, columns=COLS, index=COLS, dtype=np.float64)
    for i, row in transition_df.iterrows():
        for k, v in row.items():
            # SKIP THE FIRST AND LAST DELETE STATES
            if (k[0] == 'D' and i == 0) or (k[1] == 'D' and i == hmm.M):
                continue
                
            from_ = f'{k[0]}{i}'
            to_ = f'{k[1]}{i if k[1] in ['I'] else i + 1}'
            
            trans_matdf.loc[to_, from_] = v

    return trans_matdf

# Removing the Delete States

In [4]:
def remove_silent_states(A, state_labels, silent_labels):
    """
    Eliminate silent (non-emitting) states from a transition matrix.

    Parameters
    ----------
    A : (n x n) numpy array
        Transition matrix where A[i, j] = P(to state i | from state j),
        i.e. columns sum to 1.
    state_labels : list of str, length n
        Name of each state, in the same order as A's rows/columns.
    silent_labels : list of str
        Names of the states to eliminate (e.g. delete states).

    Returns
    -------
    A_reduced : (m x m) numpy array
        Transition matrix over only the remaining (emitting) states,
        with every silent-state detour folded into the direct
        transitions. Still column-stochastic (columns sum to 1).
    kept_labels : list of str, length m
        Names of the states that remain, in order.
    """
    silent_idx = [state_labels.index(s) for s in silent_labels]
    keep_idx = [i for i in range(len(state_labels)) if i not in silent_idx]
    kept_labels = [state_labels[i] for i in keep_idx]

    # slice the four corners
    A_EE = A[np.ix_(keep_idx, keep_idx)]      # real   -> real
    A_ED = A[np.ix_(keep_idx, silent_idx)]    # silent -> real   (exit)
    A_DE = A[np.ix_(silent_idx, keep_idx)]    # real   -> silent (entry)
    A_DD = A[np.ix_(silent_idx, silent_idx)]  # silent -> silent

    # fold all detours through the silent states back into A_EE
    n_silent = len(silent_idx)
    fundamental = np.linalg.inv(np.eye(n_silent) - A_DD)   # (I - A_DD)^-1
    correction = A_ED @ fundamental @ A_DE

    A_reduced = A_EE + correction
    return A_reduced, kept_labels

# Create the Emission Matrix

In [20]:
def create_emission_matrix(match_df, insert_df):
    length = len(insert_df) - 1
    order = ['M', 'I']
    COLS = [order[j] + f'{i+1}' for i in range(length) for j in range(2)]
    COLS.insert(0, 'M0')
    COLS.insert(1, 'I0')
    COLS.append(f'M{length+1}')
    
    emission_matdf = pd.DataFrame(0, columns=COLS, index=insert_df.columns, dtype=float)
    symbols = list(match_df.columns)
    for key in red_cols:
        if key == 'M0' or key == f'M{length+1}': continue
        for sym in symbols:
            df = insert_df if key[0] == 'I' else match_df
            emission_matdf.loc[sym, key] = df[sym][int(key[1:])]
            
    return emission_matdf

# Sample

Read the HMM file

In [21]:
hmm, match_df, insert_df, trans_df = hmm_to_dataframes('profiles/PF30808.hmm')

name = hmm.name.decode() if isinstance(hmm.name, bytes) else hmm.name
print(f"NAME: {name}   M (nodes): {hmm.M}")
print("\nMatch emissions (head):")
print(match_df.head())
print("\nInsert emissions (head):")
print(insert_df.head())
print("\nTransition probabilities (head):")
print(trans_df.head())

# remove the first row of match_df
match_df = match_df.iloc[1:]

NAME: DUF8746   M (nodes): 64

Match emissions (head):
             A         C         D         E         F         G         H  \
node                                                                         
0     1.000000  0.000000  0.000000  0.000000  0.000000  0.000000  0.000000   
1     0.093280  0.005717  0.066717  0.074059  0.006805  0.076815  0.021318   
2     0.110199  0.005729  0.044202  0.074407  0.036970  0.056293  0.032837   
3     0.055058  0.004957  0.002106  0.003331  0.008204  0.003967  0.001705   
4     0.065332  0.005094  0.028498  0.050838  0.015482  0.022231  0.022389   

             I         K         L         M         N         P         Q  \
node                                                                         
0     0.000000  0.000000  0.000000  0.000000  0.000000  0.000000  0.000000   
1     0.011386  0.063037  0.029404  0.011564  0.057014  0.081211  0.063514   
2     0.017471  0.060860  0.026234  0.012165  0.032542  0.014848  0.047115   
3     0.

Create the transition matrix

In [22]:
trans_matdf = create_transition_matrix(trans_df)

Remove the silent (delete) states in transition matrix

In [23]:
cols = list(trans_matdf.columns)
d_cols = [i for i in cols if i[0] == 'D']
trans_matrix, red_cols = remove_silent_states(np.array(trans_matdf), cols, d_cols)
transition_df = pd.DataFrame(trans_matrix, columns=red_cols, index=red_cols)

Create the emission matrix

In [27]:
emission_df = create_emission_matrix(match_df, insert_df)

Double Check

In [28]:
transition_df.to_csv('transition.csv')
emission_matdf.to_csv('emission.csv')